# Explicit disaggregation workflow

A small working example: **3 source regions → 2 model regions**, with **2 original sectors + 1 new sector**. Each workflow stage has its own API call. Data and results are synthetic.

Use **`codex/explicit-workflow` in both MARIO and Database disaggregation**, with these local checkouts importable in the notebook environment. CVXLab and CLARABEL must be installed. No SharePoint files or commercial solver are needed. The existing workshop notebook is unchanged.

In [ ]:
from pathlib import Path
import tempfile
import pandas as pd
import mario
from database_disaggregation import Geography, workflow
from IPython.display import display, FileLink

work = Path(tempfile.mkdtemp(prefix="explicit-disaggregation-"))

## 1. Source database

For this portable example we create a balanced monetary IOT in memory. With real data, replace this cell with the appropriate MARIO parser: all following APIs consume the same MARIO database object.

In [ ]:
axis = pd.MultiIndex.from_product(
    [["R1", "R2", "R3"], ["Sector"], ["Parent", "Stable"]], names=["Region", "Level", "Item"])
demand = pd.MultiIndex.from_product(
    [["R1", "R2", "R3"], ["Consumption category"], ["HH"]], names=axis.names)
units = {kind: pd.DataFrame({"unit": [unit] * len(items)}, index=items)
         for kind, items, unit in [("Sector", ["Parent", "Stable"], "M USD"),
             ("Factor of production", ["VA"], "M USD"), ("Satellite account", ["CO2"], "tonne")]}
native_db = mario.Database(name="Small workflow example", table="IOT", units=units,
    Z=pd.DataFrame(1., index=axis, columns=axis), Y=pd.DataFrame(4., index=axis, columns=demand),
    V=pd.DataFrame(12., index=["VA"], columns=axis), E=pd.DataFrame(1., index=["CO2"], columns=axis),
    EY=pd.DataFrame(0., index=["CO2"], columns=demand))

## 2. Collected data

This cell creates a tiny synthetic collection using MARIO's Excel template. The new sector **A** belongs to **Parent**, produces 6 M USD in each source region and uses the same simple cost structure everywhere. These lines only construct the example inputs; replace `collection` with your own resolved inventory directory for real data.

In [ ]:
collection = work / "collection"
collection.mkdir()
source = collection / "A.xlsx"
regions = native_db.get_index("Region")
native_db.get_add_sectors_excel(path=source, items=["A"], regions=regions)
master = pd.read_excel(source, sheet_name="Master").astype(object)
master["Parent Sector"], master["Add or Split"], master["Unit"] = "Parent", "Split", "M USD"
profile = pd.DataFrame(
    [[1/9, "M USD", "Parent", "Sector", "Parent", r, "Update"] for r in regions]
    + [[2/3, "M USD", "VA", "Factor of production", "VA", None, "Update"]],
    columns=["Quantity", "Unit", "Input", "Item type", "DB Item", "DB Region", "Change type"])
with pd.ExcelWriter(source, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    master.to_excel(writer, sheet_name="Master", index=False)
    for sheet in master["Inventory sheet"]:
        profile.to_excel(writer, sheet_name=sheet, index=False)
    pd.DataFrame([["A", r, 6., "M USD", "Synthetic", None] for r in regions],
        columns=["Sector", "Region", "Quantity", "Unit", "Source", "Notes"]).to_excel(writer, sheet_name="Total outputs", index=False)
    pd.DataFrame([["A", "R1", "R2", 1., "M USD", "Synthetic", None], ["A", "R1", "R3", 2., "M USD", "Synthetic", None],
                  ["A", "R2", "R3", 3., "M USD", "Synthetic", None]],
        columns=["Sector_from", "Region_from", "Region_to", "Quantity", "Unit", "Source", "Notes"]).to_excel(writer, sheet_name="Trades", index=False)
    pd.DataFrame({"tol_Name": ["delta", "eps"], "values": [0., 0.]}).to_excel(
        writer, sheet_name="Tolerances", index=False)

## 3. Prepare inventories and aggregate the database

Use one explicit regional mapping for both. Preparation reuses the repository's existing inventory handling. It writes a MARIO-readable workbook, without solving or saving the source database. For the real extended ENTICE collection, `workflow.prepare_model_inputs(...)` exposes the existing project recipe.

In [ ]:
geography = Geography({"R1": "G", "R2": "G", "R3": "H"})
prepared = workflow.prepare_inputs(collection, work / "prepared", database=native_db, geography=geography)
db = native_db.aggregate(None, levels="Region", region_aggregation=geography.mapping,
                         zero_output_epsilon=None, inplace=False)

## 4. Build the optimization model

Build the CVXLab/CVXPY problem and its inputs. No solver runs in this cell. This example uses one quadratic optimization for the whole workbook. Grouped execution still uses MARIO's checkpoint controller; it repeats build, solve and assembly for each group.

In [ ]:
model = workflow.build_model(db, prepared, directory=work / "model",
    split_model_options={"model": "Split_sectors_MRIO_quadratic", "accounting": "inherit",
                         "compact_constraints": True, "scale": 10.})
print(model.status)

## 5. Solve the model

Run the solver and the existing numerical validation. The validated solution is available; the new MARIO database has not been assembled yet.

In [ ]:
solution = workflow.solve_model(model, solver="CLARABEL")
print(solution.solver_status)

## 6. Assemble the disaggregated database

Reconstruct the MARIO matrices and promote the solution to baseline. No solver runs here. The original `db` and `native_db` remain unchanged.

In [ ]:
result = workflow.assemble_database(solution)
print(result.get_index("Region"), result.get_index("Sector"))

## 7. Export a new run

Save baseline flows, independent output X, the input workbook and run metadata. Existing run directories cannot be overwritten. `workflow.load_database(exported)` verifies the files and reloads the saved result; this exported run is distinct from a resumable sequential checkpoint.

In [ ]:
exported = workflow.export_database(result, work / "database", inputs=prepared)

## 8. Prepare the dashboard

Reuse the common analysis and dashboard code. The archived workbook supplies sector identities and production/trade checks. No external comparison databases are attached to this synthetic example.

In [ ]:
dashboard = workflow.build_dashboard(exported, work / "dashboard.html")
display(FileLink(dashboard))

After preparation, `workflow.run(...)` executes the same model, solve, assembly, export and dashboard APIs in order. Keep the explicit calls when showing the workflow step by step.